# Week 13: Advanced Dynamic Programming (Interval DP, LIS, Bitmask TSP)

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Matrix Chain Multiplication, O(n log n) LIS, Rod Cutting, and Held-Karp exact TSP.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-13/labs/W13_Advanced_Dynamic_Programming.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Interval DP (Matrix Chain Multiplication) & $O(n \log n)$ LIS

Matrix Chain Multiplication searches over interval lengths and split indices $k \in [i, j-1]$ in $O(n^3)$ time. Patience sorting computes LIS length in $O(n \log n)$ via binary search (`bisect_left`).

In [1]:
import bisect
import math


def matrix_chain_order(dims):
    n = len(dims) - 1
    dp = [[0] * n for _ in range(n)]
    split = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i][j] = math.inf
            for k in range(i, j):
                cost = dp[i][k] + dp[k + 1][j] + dims[i] * dims[k + 1] * dims[j + 1]
                if cost < dp[i][j]:
                    dp[i][j], split[i][j] = cost, k

    def fmt(i, j):
        return f"A{i+1}" if i == j else f"({fmt(i, split[i][j])} x {fmt(split[i][j]+1, j)})"

    return int(dp[0][n - 1]), fmt(0, n - 1)


def lis_nlogn(seq):
    tails = []
    for x in seq:
        i = bisect.bisect_left(tails, x)
        if i == len(tails): tails.append(x)
        else: tails[i] = x
    return len(tails)


cost, expr = matrix_chain_order([30, 35, 15, 5, 10, 20, 25])
assert cost == 15125
assert lis_nlogn([10, 9, 2, 5, 3, 7, 101, 18]) == 4
print(f"Optimal MCM cost = {cost}: {expr}")
print("LIS([10, 9, 2, 5, 3, 7, 101, 18]) =", lis_nlogn([10, 9, 2, 5, 3, 7, 101, 18]))


Optimal MCM cost = 15125: ((A1 x (A2 x A3)) x ((A4 x A5) x A6))
LIS([10, 9, 2, 5, 3, 7, 101, 18]) = 4


## 2. Bitmask DP: Held-Karp Exact TSP Tour

State `(mask, u)` tracks the shortest path visiting all vertices in bitmask `mask` and ending at `u`, reducing $O(n!)$ brute-force tour search to $O(n^2 2^n)$.

In [2]:
def held_karp_tsp(dist):
    n = len(dist)
    full = 1 << n
    dp = [[math.inf] * n for _ in range(full)]
    dp[1][0] = 0.0
    for mask in range(1, full):
        if not (mask & 1):
            continue
        for u in range(n):
            if dp[mask][u] == math.inf:
                continue
            for v in range(1, n):
                if not (mask & (1 << v)):
                    nxt = mask | (1 << v)
                    dp[nxt][v] = min(dp[nxt][v], dp[mask][u] + dist[u][v])
    return min(dp[full - 1][u] + dist[u][0] for u in range(1, n))


tour = held_karp_tsp([[0, 10, 15, 20], [10, 0, 35, 25], [15, 35, 0, 30], [20, 25, 30, 0]])
assert tour == 80.0
print("Optimal 4-stop Zanzibar depot tour cost:", tour)


Optimal 4-stop Zanzibar depot tour cost: 80.0
